# SOLUCIÓN DEL EJERCICIO

Actualizar el proyecto del día 1 para resumir una página web para utilizar un modelo de código abierto que se ejecuta localmente a través de Ollama en lugar de OpenAI.

Podrás utilizar esta técnica para todos los proyectos posteriores si prefieres no utilizar APIs de pago.

**Ventajas
1. No hay cargos por API - de código abierto
2. Los datos no salen de tu caja

**Desventajas
1. 1. Mucho menos potente que el modelo Frontier

## Recapitulación sobre la instalación de Ollama

Basta con visitar [ollama.com](https://ollama.com) e instalar.

Una vez completado, el servidor ollama ya debería estar funcionando localmente.  
Si visita  
[http://localhost:11434/](http://localhost:11434/)

Deberías ver el mensaje `Ollama se está ejecutando`.  

Si no es así, abre un nuevo Terminal (Mac) o Powershell (Windows) e introduce `ollama serve`.  
A continuación, inténtalo de nuevo con [http://localhost:11434/](http://localhost:11434/).


In [1]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [2]:
# Constantes

MODEL = "llama3.2"

In [3]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Connec

## Tipos de indicaciones/prompts

Quizás ya sepas esto, pero si no, te resultará muy familiar.

Los modelos como GPT4o han sido entrenados para recibir instrucciones de una manera particular.

Esperan recibir:

**Una indicación del sistema** que les indique qué tarea están realizando y qué tono deben usar

**Una indicación del usuario**: el inicio de la conversación al que deben responder

In [5]:
# Define nuestro mensaje de sistema: puedes experimentar con esto más tarde, cambiando la última oración a "Responder en Markdown en español".

system_prompt = "Eres un asistente que analiza el contenido de un sitio web \
y proporciona un breve resumen, ignorando el texto que podría estar relacionado con la navegación. \
Responder en Markdown."

In [6]:
# Una función que escribe un mensaje de usuario que solicita resúmenes de sitios web:

def user_prompt_for(website):
    user_prompt = f"Estás viendo un sitio web titulado {website.title}"
    user_prompt += "\nEl contenido de este sitio web es el siguiente; \
    proporciona un breve resumen de este sitio web en formato Markdown. \
    Si incluye noticias, productos o anuncios, resúmelos también.\n\n"
    user_prompt += website.text
    return user_prompt

## Mensajes

La API de OpenAI espera recibir mensajes en una estructura particular.
Muchas de las otras API comparten esta estructura:

```
[
    {"role": "system", "content": "el mensaje de sistema va aquí"},
    {"role": "user", "content": "el mensaje de usuario va aquí"}
]

In [7]:
# Puedes ver cómo esta función crea exactamente el formato anterior

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Es hora de unirlo todo: ahora con Ollama en lugar de OpenAI

In [8]:
# Y ahora: llamamos a la función Ollama en lugar de OpenAI

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [9]:
summarize("https://edwarddonner.com")

'**Resumen del sitio web**\n\nEl sitio web titulado "Home - Edward Donner" es la página principal de un blog personal y profesional de Edward Donner, el co-fundador y CTO de Nebula.io. El contenido se centra en tecnología de inteligencia artificial (IA) y su aplicación en diferentes áreas.\n\n**Noticias**\n\n* El sitio web destaca eventos y publicaciones relacionadas con la IA, como:\n + "AI in Production: Gen AI and Agentic AI on AWS at scale" (15 de septiembre de 2025)\n + "Connecting my courses – become an LLM expert and leader" (28 de mayo de 2025)\n + "2025 AI Executive Briefing" (21 de abril de 2025)\n\n**Productos o servicios**\n\n* Nebula.io, la empresa fundada por Edward Donner, ofrece una plataforma que aplica IA para ayudar a las personas a descubrir su potencial y pursue sus razones de ser.\n* El sitio web también menciona el producto "untapt", una startup de AI adquirida en 2021.\n\n**Anuncios**\n\n* El sitio web incluye un llamado a la acción para conectarse con Edward Do

In [10]:
# Una función para mostrar esto de forma clara en la salida de Jupyter, usando markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [11]:
display_summary("https://edwarddonner.com")

**Resumen del sitio web Home - Edward Donner**

El sitio web de Edward Donner es una plataforma que combina la tecnología artificial (TA) con el empoderamiento personal. El creador, Ed, es un experto en LLMs (modelos de lenguaje profundos) y co-fundador del servicio Nebula.io.

**Noticias**

* En septiembre de 2025, Edward Donner presentará una charla sobre "AI in Production: Gen AI and Agentic AI on AWS at scale".
* En mayo de 2025, ofrecerá un curso en línea sobre "Connecting my courses – become an LLM expert and leader".
* El 21 de abril de 2025, presentó el "2025 AI Executive Briefing".

**Productos y servicios**

El sitio web menciona la plataforma Nebula.io, que utiliza LLMs para ayudar a las personas a descubrir su potencial y propósito. La plataforma es utilizada por reclutadores para encontrar talento y ha sido patentada.

**Intereses personales**

* Edward Donner es un entusiasta de la música electrónica y DJ.
* Está conectado con la comunidad del Hacker News y disfruta de los artículos que comparten en esta plataforma.

**Conectarse con Ed**

Puedes contactar a Ed mediante su correo electrónico `ed [at] edwarddonner [dot] com` o siguiéndolo en sus redes sociales:

* LinkedIn
* Twitter
* Facebook

También puedes suscribirte a su boletín.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [12]:
display_summary("https://cnn.com")

**Resumen del sitio web de CNN**

El sitio web de CNN es un canal de noticias en línea que ofrece información actualizada sobre eventos internacionales, políticos, deportes, entretenimiento y más. El sitio web está dividido en secciones diferentes, como:

* **Noticias**: La sección principal donde se publican artículos y videos sobre los últimos acontecimientos mundiales.
* **Videos**: Un espacio dedicado a los videos de noticias, entrevistas y reportajes.
* **Espectáculos**: Una sección que cubre eventos del entretenimiento, como películas, televisión y música.
* **Deportes**: Sección donde se publican noticias y resultados deportivos.

**Anuncios y productos**

En el sitio web de CNN, se pueden encontrar anuncios para productos y servicios relacionados con la tecnología, la salud, la finanza y más. Algunos ejemplos de anuncios que se encuentran en el sitio web son:

* Anuncios de productos de tecnología
* Anuncios de productos de salud y bienestar
* Anuncios de servicios financieros
* Anuncios de viajes y turismo

**Noticias importantes**

Algunas noticias importantes que se publicaron en el sitio web de CNN incluyen:

* El plan de cesaría de Trump para la guerra en Gaza
* La situación política en Venezuela
* Los resultados del super bowl
* Las últimas noticias sobre la pandemia de COVID-19

**Funcionalidades**

El sitio web de CNN ofrece varias funcionalidades, como:

* **Transmisión en vivo**: Permite ver transmisiones en vivo de eventos y noticias.
* **Notificaciones**: Permite recibir notificaciones de noticias y eventos importantes.
* **Seguimiento de noticias**: Permite seguir las últimas noticias sobre temas específicos.

En resumen, el sitio web de CNN es una fuente de información en línea que ofrece noticias, videos y análisis sobre los últimos acontecimientos mundiales.

In [13]:
display_summary("https://anthropic.com")

# Resumen del sitio web Anthropic

El sitio web de Anthropic es un plataforma de inteligencia artificial (IA) dedicada a la seguridad y el bienestar humano. Aquí te presento los puntos clave:

## Misión y enfoque
- **Público benefit corporation**: Anthropic se enfoca en asegurar los beneficios de la IA y mitigar sus riesgos.
- **Desarrollo responsable**: El sitio destaca la importancia del desarrollo responsable de las tecnologías de IA.

## Productos y servicios

* **Claude**: Un modelo de lenguaje de grandes cantidades de datos (LLMD) diseñado para agentes, coding y uso computacional.
* **Opus**: Un modelo de lenguaje basado en opacidad ( opacity-based model).
* **Sonnet**: Un modelo de lenguaje basado en Sonnets.
* **Haiku**: Un modelo de lenguaje basado en Haikus.

## Recursos

- **Anthropic Academy**: Acceso a cursos y recursos para aprender a construir con Claude.
- **Developer docs**: Documentación para desarrolladores sobre la plataforma de desarrollo de Claude.
- **Courses**: Cursos en línea para aprender sobre AI, coding y desarrollo.

## Noticias y anuncios

* **Claude Sonnet 4.5**: Introducción del mejor modelo de agentes, código y uso computacional.
* **Claude Opus 4.1**: Actualización del modelo de lenguaje Opus.
* **Project Vend**: Un proyecto relacionado con la seguridad y responsabilidad en el desarrollo de IA.
* **Tracing the thoughts of a large language model**: Artículo sobre interpretabilidad y visibilidad en modelos de lenguaje.

## Políticas

- **Responsible Scaling Policy**: Una política para asegurar una escala responsable de las tecnologías de IA.
- **Trust center**: Un centro de confianza que ofrece información sobre la seguridad y responsabilidad de Anthropic.

## Contacto
- **Try Claude**: Descarga del modelo de lenguaje Claude.
- **Pricing**: Información sobre los planes de precios para la plataforma de desarrollo de Claude.
- **Contact sales**: Contacto con las ventas de Anthropic.

# Compartir tu código

¡Me encantaría que compartieras tu código después para que yo pueda compartirlo con otros! Notarás que algunos estudiantes ya han realizado cambios (incluida una implementación de Selenium) que encontrarás en la carpeta community-contributions. Si deseas agregar tus cambios a esa carpeta, envía una solicitud de incorporación de cambios con tus nuevas versiones en esa carpeta y yo fusionaré tus cambios.

Si no eres un experto en Git (¡y yo no lo soy!), entonces GPT ha dado algunas instrucciones útiles sobre cómo enviar una solicitud de incorporación de cambios. Es un proceso un poco complicado, pero una vez que lo hayas hecho una vez, estará bastante claro. Como consejo profesional: es mejor si borras las salidas de tus cuadernos Jupyter (Editar >> Limpiar las salidas de todas las celdas y luego Guardar) para tener cuadernos limpios.

Instrucciones de relaciones públicas cortesía de un amigo de IA: https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c